In [51]:
%pip install pandas numpy matplotlib seaborn scikit-learn xgboost jupyterlab kagglehub


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)   # показывать все колонки
sns.set_style('whitegrid')                 # приятный стиль графиков
print("Библиотеки загружены")

Библиотеки загружены


In [53]:
df = pd.read_csv('contracts.csv', low_memory=False)
print("Размер датасета:", df.shape)
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'contracts.csv'

In [ ]:
# single_bidder = 1, если в тендере был ровно один участник (Offers_Number == 1)
# Строки, где Offers_Number пустой (34 шт.), убираем — без таргета они бесполезны
df = df.dropna(subset=['Offers_Number']).copy()
df['single_bidder'] = (df['Offers_Number'] == 1).astype(int)

print("Распределение классов:")
print(df['single_bidder'].value_counts())
print("\nДоля single-bidder тендеров:",
      round(df['single_bidder'].mean() * 100, 1), "%")

In [ ]:
# Это признаки, которые становятся известны ТОЛЬКО ПОСЛЕ присуждения тендера.
# Использовать их = "подсматривать ответ". Обязательно убираем.
leakage_cols = [
    'Winner', 'Winner_VAT', 'Winner_Country', 'Winner_City', 'Winner_Address',
    'Value', 'Value_RON', 'Value_EUR',          # фактическая сумма (известна после)
    'Award_Anouncement_Number', 'Award_Announcement_Date',
    'Contract_Number', 'Contract_Date', 'Contract_Title',
    'Contract_Conclusion_Type', 'Subcontracted',
    'Offers_Number',                            # из него сделан таргет
]


In [ ]:
# Колонки, где пусто больше 60% — пользы мало, только шум
high_null_cols = [
    'Periodic_Contract', 'EU_Fund', 'EU_Funds', 'With_Electronic_Auction',
    'Financing_Type', 'Garantee_Deposits', 'Financing_Method',
    'Contracting_Authority_Type', 'Legislation_Type_ID',
    'Participation_Announcement_Number', 'Participation_Announcement_Date',
    'CPV_Code_ID',                              # дубликат CPV_Code (числовой id)
    'Contracting_Authority_VAT',               # просто идентификатор, не признак
    # Participation_Estimated_Value — в разных валютах (EUR/RON), несопоставимо.
    # Для простоты убираем. При желании можно вернуть, приведя всё к одной валюте.
    'Participation_Estimated_Value', 'Participation_Estimated_Value_Currency',
]

cols_to_drop = leakage_cols + high_null_cols
df_clean = df.drop(columns=cols_to_drop)
print("Колонок было:", df.shape[1])
print("Колонок осталось:", df_clean.shape[1])
print("\nОставшиеся колонки:")
print(df_clean.columns.tolist())


In [ ]:
# 887к строк — многовато для быстрых экспериментов на слабой машине.
# Берём случайную выборку 150к строк. Это полностью репрезентативно
# и ускорит обучение в разы. random_state=42 — для воспроизводимости.
df_sample = df_clean.sample(n=150_000, random_state=42).reset_index(drop=True)
print("Размер рабочей выборки:", df_sample.shape)
print("Доля single-bidder в выборке:",
      round(df_sample['single_bidder'].mean() * 100, 1), "%")


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_sample, x='single_bidder',
              hue='single_bidder', palette=['#4C72B0', '#DD8452'], legend=False)
plt.title('Распределение классов: single-bidder vs конкурентные')
plt.xlabel('single_bidder (0 = конкурентный, 1 = единственный участник)')
plt.ylabel('Количество тендеров')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120)   # сохраняем для отчёта
plt.show()

In [ ]:
# Гипотеза: неконкурентные процедуры (Negociere fara anunt) дают больше
# тендеров с одним участником. Проверим.
proc_stats = (df_sample.groupby('Procedure_Type')['single_bidder']
              .agg(['mean', 'count'])
              .sort_values('mean', ascending=False))
proc_stats['mean'] = (proc_stats['mean'] * 100).round(1)
proc_stats.columns = ['Доля single-bidder %', 'Кол-во тендеров']
print(proc_stats)


In [ ]:
print("Пропуски в рабочей выборке:")
nulls = df_sample.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print(nulls if len(nulls) else "Пропусков нет")

In [29]:
# Сохраняем подготовленную выборку, чтобы завтра не грузить заново
df_sample.to_csv('tenders_clean.csv', index=False)
print("Готово! Сохранено в tenders_clean.csv")
print("Размер:", df_sample.shape)


Готово! Сохранено в tenders_clean.csv
Размер: (150000, 9)
